# X Sentiment Analysis

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [57]:
df = pd.read_csv('train.txt', sep= ';', header = None, names = ['text', 'emotion'] )

In [58]:
df.sample(5)

,text,emotion
6788,i feel a bit like a naughty child because i wa...,love
11270,i pray that they will continue to be giving co...,love
13527,i finished blogging i was feeling shaky and ch...,fear
5417,i was feeling a bit homesick so i made a last ...,sadness
9807,i have my favorite cookies in the house oatmea...,sadness


In [59]:
df.isnull().sum()

,0
text,0
emotion,0


In [60]:
unique_emotion = df['emotion'].unique()
emotion_num = {}
i = 0
for emotion in unique_emotion:
  emotion_num[emotion] = i
  i += 1

df['emotion'] = df['emotion'].map(emotion_num)

In [61]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [62]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [63]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('', '', string.punctuation))


In [64]:
df['text']= df['text'].apply(remove_punc)

In [65]:
def remove_num(text):
  new = ""
  for char in text:
    if not char.isdigit():
      new += char
  return new

In [66]:
df['text'] = df['text'].apply(remove_num)

In [67]:
def remove_emojis(txt):
  new = ''
  for i in txt:
    if i.isascii():
      new += i
  return new

In [68]:
df['text'] = df['text'].apply(remove_emojis)

In [69]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [70]:
stop_words = set(stopwords.words('english'))


In [71]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [72]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [73]:
df['text'] = df['text'].apply(remove)

In [74]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [75]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)


In [77]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


In [78]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [79]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [80]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [81]:

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [82]:
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


In [83]:
from sklearn.linear_model import LogisticRegression


In [84]:
logistic_model = LogisticRegression(max_iter=1000)


In [85]:
logistic_model.fit(X_train_tfidf,y_train)


LogisticRegression(max_iter=1000)

In [86]:
log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test,log_pred ))

0.8628125


In [87]:
# 1. Linear Support Vector Machine (LinearSVC) — Best & Fastest Classical ML
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

svm_model = LinearSVC(C=1.0, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

SVM Accuracy: 0.891875
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.88      0.88       427
           2       0.83      0.74      0.78       296
           3       0.88      0.70      0.78       113
           4       0.87      0.85      0.86       397
           5       0.89      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [89]:
# LightGBM / XGBoost
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05)
lgbm.fit(X_train_tfidf, y_train)

print("LightGBM Accuracy:", accuracy_score(y_test, lgbm.predict(X_test_tfidf)))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.243307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 28066
[LightGBM] [Info] Number of data points in the train set: 12800, number of used features: 1218
[LightGBM] [Info] Start training from score -1.235722
[LightGBM] [Info] Start training from score -2.000168
[LightGBM] [Info] Start training from score -2.541477
[LightGBM] [Info] Start training from score -3.328150
[LightGBM] [Info] Start training from score -2.117663
[LightGBM] [Info] Start training from score -1.081340


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Accuracy: 0.8625


In [88]:
# TfidfVectorizer ko n-gram aur sublinear scaling ke saath update karein
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=15000, sublinear_tf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Phir LinearSVC ya LogisticRegression run karein
svm_model.fit(X_train_tfidf, y_train)
print("Tuned Accuracy:", accuracy_score(y_test, svm_model.predict(X_test_tfidf)))

Tuned Accuracy: 0.90375
